# Diabetes mellitus prediction

This notebook uses the classes of the `diabetes_prediction` library to load the patients data, preprocess it, create the features, train a model and predict the probabilities on the test set.

In [1]:
from sklearn.metrics import roc_auc_score

from diabetes_prediction.data import CSVReader
from diabetes_prediction.features import BinaryFeature, OneHotEncodingFeature
from diabetes_prediction.models import RandomForestModel
from diabetes_prediction.preprocessing import MeanImputer, MissingRowsRemover

ModuleNotFoundError: No module named 'diabetes_prediction'

In [ ]:
DATA_PATH = "../sample_diabetes_mellitus_data.csv"

REQUIRED_COLUMNS = ["age", "gender", "ethnicity"]
MEAN_FILL_COLUMNS = ["height", "weight"]
TARGET_COLUMN = "diabetes_mellitus"

## 1. Load the data and split it between train and test

In [ ]:
reader = CSVReader(DATA_PATH)
train_df, test_df = reader.read()
train_df.shape, test_df.shape

((8000, 52), (2000, 52))

## 2. Preprocess the data

- Remove the rows with NaN in age, gender and ethnicity.
- Fill the NaN in height and weight with the mean.

Each preprocessor is fitted on the **train** set only, so no information from the test set is used.

In [ ]:
preprocessors = [
    MissingRowsRemover(REQUIRED_COLUMNS),
    MeanImputer(MEAN_FILL_COLUMNS),
]

for preprocessor in preprocessors:
    preprocessor.fit(train_df)
    train_df = preprocessor.transform(train_df)
    test_df = preprocessor.transform(test_df)

train_df.shape, test_df.shape

((7497, 52), (1871, 52))

## 3. Create the features

- Gender as a binary variable (F = 1, M = 0).
- One hot encoding of ethnicity.

In [ ]:
features = [
    BinaryFeature("gender", positive_value="F"),
    OneHotEncodingFeature("ethnicity"),
]

for feature in features:
    feature.fit(train_df)
    train_df = feature.transform(train_df)
    test_df = feature.transform(test_df)

ethnicity_columns = [column for column in train_df.columns if column.startswith("ethnicity_")]
FEATURE_COLUMNS = ["age", "height", "weight", "gender"] + ethnicity_columns
FEATURE_COLUMNS

['age',
 'height',
 'weight',
 'gender',
 'ethnicity_African American',
 'ethnicity_Asian',
 'ethnicity_Caucasian',
 'ethnicity_Hispanic',
 'ethnicity_Native American',
 'ethnicity_Other/Unknown']

## 4. Train the model

`max_depth` limits the trees so the random forest does not memorise the train set.

In [ ]:
model = RandomForestModel(
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    hyperparameters={"max_depth": 5, "random_state": 42},
)
model.train(train_df)

## 5. Predict the probabilities on the test set

`predict` returns one column per class, so we keep the probability of the positive class (1).

In [ ]:
test_df["predictions"] = model.predict(test_df)[:, 1]
test_df[[TARGET_COLUMN, "predictions"]].head()

,diabetes_mellitus,predictions
4765,0,0.301200
9601,0,0.186065
4570,0,0.134352
4320,1,0.253245
2726,0,0.305600


## 6. Test ROC AUC

In [ ]:
test_roc_auc = roc_auc_score(test_df[TARGET_COLUMN], test_df["predictions"])
print(f"Test ROC AUC: {test_roc_auc:.3f}")

Test ROC AUC: 0.687
